# Setup

## Imports

In [24]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from pathlib import PurePosixPath
import sqlite3
from pathlib import Path
from tqdm.auto import tqdm
import time 

## Definitions

In [25]:
headers = {"User-Agent": "Mozilla/5.0"}
master_list = ["https://astrolab.awh.durham.ac.uk/east_16/east-16_logs_2026.html", "https://astrolab.awh.durham.ac.uk/draco2/draco2_logs_2026.html",
               "https://astrolab.awh.durham.ac.uk/far_east_20/far-east-20_logs_2026.html", "https://astrolab.awh.durham.ac.uk/west_14/west-14_logs_2026.html",
               "https://astrolab.awh.durham.ac.uk/pt5m/pt5m_logs_2026.html",
              ]
   #             "telescope1_2026.html", "telescope2_2026.html", 
    #           "telescope3_2026.html", "telescope4_2026.html", 
     #          "telescope5_2026.html", "telescope6_2026.html",
      #        )

## SQLite Setup

In [26]:
db_path = Path("../databases/Durham_Observation_Database.db").resolve()
conn = sqlite3.connect(db_path)

conn.execute("""
   CREATE TABLE IF NOT EXISTS observations (
       img_num TEXT NOT NULL,
       object TEXT,
       ra REAL,
       dec REAL,
       A TEXT,
       start_time TEXT,
       filter TEXT,
       exp_time REAL,
       fwhm REAL,
       telescope TEXT,
       obs_date TEXT,
       url_source TEXT NOT NULL,
       UNIQUE (url_source, img_num)
   )
""")




conn.execute("""
   CREATE TABLE IF NOT EXISTS processed_logs (
       url_source TEXT PRIMARY KEY,
       processed_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
   )
""")

# Functions

### Function 1
This function finds all of the daily observation links on a given telescope's year page and saves those links to a list.

In [27]:
def year_scrape(url, headers):
    year_link_list = []
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    

    year_soup = BeautifulSoup(response.text, "html.parser")
    links = year_soup.find_all("a", href=True)

    for link in links:
        year_link_list.append(urljoin(url, link["href"]))


    
    return(year_link_list)

### Function 2

In [28]:
def parse_daily_log(html, url):

    # set up soup and rows
    soup = BeautifulSoup(html, "html.parser")
    page_rows = soup.find_all("tr")


    # append each row to get non labeled table
    unlabeled_table=[]
    for row in page_rows:
        cells = row.find_all("td")
        values = [cell.get_text(strip=True) for cell in cells]

        if len(values) != 9:
            continue

        if not values[0].isdigit():
            continue

        unlabeled_table.append(values)


    # set up observation dictionary (labeled one)
    labeled_table = []
    path = PurePosixPath(urlparse(url).path)
    obs_date = path.stem.replace("_", "-")
    telescope_names = {
        "east_16": "east-16",
        "west_14": "west-14",
        "far_east_20": "far-east-20",
    }
    raw_telescope = path.parts[-3]
    telescope = telescope_names.get(raw_telescope, raw_telescope)
    
    #the actual obs dict
    for position in unlabeled_table:
        observation = {
            "img_num": position[0],
            "object": position[1],
            "ra": float(position[2]),
            "dec": float(position[3]),
            "A": position[4],
            "start_time": position[5],
            "filter": position[6],
            "exp_time": float(position[7]),
            "fwhm": optional_float(position[8]),    #with the new float func.
            "telescope": telescope,
            "obs_date": obs_date,
            "url_source": url,
        }
        labeled_table.append(observation)
    return labeled_table


#to fix the fwhm values
def optional_float(text):
    text = text.strip()

    if text in {"", "-"}:
        return None

    return float(text)

### Function 3
Checks if a given url has already been processed

In [29]:
def is_log_processed(conn, url):
   result = conn.execute("""
       SELECT 1
       FROM processed_logs
       WHERE url_source = ?
   """, (url,)).fetchone()

   return result is not None

### Function 4

In [30]:
def save_daily_log(conn, url, observations):
   with conn:
       conn.executemany("""
           INSERT INTO observations (
               img_num, object, ra, dec, A, start_time,
               filter, exp_time, fwhm, telescope, obs_date, url_source
           )
           VALUES (
               :img_num, :object, :ra, :dec, :A, :start_time,
               :filter, :exp_time, :fwhm, :telescope, :obs_date, :url_source
           )
           ON CONFLICT (url_source, img_num) DO NOTHING
       """, observations)

       conn.execute("""
           INSERT INTO processed_logs (url_source)
           VALUES (?)
           ON CONFLICT (url_source) DO NOTHING
       """, (url,))

### Main Function

In [31]:
def update_database(url_list):
    total_observations = 0
    for item in url_list:
        telescope = PurePosixPath(urlparse(item).path).parent.name
        year_list = year_scrape(item, headers)
    
        list1 = []
        for item in year_list:
            if is_log_processed(conn, item):
                continue
            else:
                list1.append(item)
    
        for link in tqdm(list1, desc=f"{telescope} nightly logs"):
            
            raw_html = requests.get(link, headers=headers, timeout=20)
            raw_html.raise_for_status()
            
            daily_log = parse_daily_log(raw_html.text, link)

            if not daily_log:
               raise ValueError(f"{link} failed")
            
            save_daily_log(conn, link, daily_log)
            total_observations += len(daily_log)
            time.sleep(1)
    print("Observations processed:", total_observations)

### In Case of Missing Pages

In [32]:
missing_url = (
       "https://astrolab.awh.durham.ac.uk/"
       "pt5m/pt5m_2026/2026_06_15.html"
   )

with conn:
   conn.execute("""
       INSERT INTO processed_logs (url_source)
       VALUES (?)
       ON CONFLICT (url_source) DO NOTHING
   """, (missing_url,))

# Actual Update

In [36]:
update_database(master_list)

east_16 nightly logs: 0it [00:00, ?it/s]

draco2 nightly logs: 0it [00:00, ?it/s]

far_east_20 nightly logs: 0it [00:00, ?it/s]

west_14 nightly logs: 0it [00:00, ?it/s]

pt5m nightly logs: 0it [00:00, ?it/s]

Observations processed: 0
